# 02 — Prédire l'insatisfaction à l'atterrissage

Responsable : **Jeanne** — branche `modeles`

Données : **train uniquement**, comparaison des modèles en validation croisée.

`test.csv` n'est **jamais** lu ici : voir le notebook 03.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # pour importer src/ depuis notebooks/

from src.data import load_train, TARGET, RANDOM_STATE
from src.features import LANDING_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.economics import gain_net, BREAK_EVEN_PRECISION

df = load_train()
X = df[LANDING_FEATURES]
y = df[TARGET]
X.shape, y.mean()

((103904, 7), np.float64(0.5666673082845703))

## 1. Variables connues à l'atterrissage

Le modèle est appliqué à **tous les passagers arrivants**, y compris ceux qui n'ont jamais répondu à l'enquête. Il ne peut donc utiliser que des informations que la compagnie possède **au moment de l'atterrissage**.

| Variable | Connue à l'atterrissage ? | Source | Décision |
|---|---|---|---|
| Classe | oui | billet | **gardée** |
| Âge | oui | réservation | **gardée**, mais les mineurs sont exclus du signalement (partie 9) |
| Type de client (fidèle / non fidèle) | oui, si c'est le statut du programme de fidélité (hypothèse) | programme de fidélité | **gardée** |
| Distance du vol | oui | plan de vol | **gardée** |
| Retard au départ | oui | opérations | **gardée** |
| Retard à l'arrivée | oui, connu dès l'atterrissage | opérations | **gardée** |
| Type de voyage (pro / perso) | **incertain** : le motif vient probablement de l'enquête | enquête | **testée avec et sans** (partie 7) ; modèle retenu **sans** |
| Genre | oui | civilité de la réservation | **exclue** : quasi aucun écart d'insatisfaction (57,3 % / 56,1 %), risque de discrimination |
| 14 notes de service | **non** : elles n'existent qu'après l'enquête | enquête | **exclues** du modèle (utilisées seulement pour expliquer la satisfaction, notebook 01) |
| `id` | — | — | **exclue** : identifiant technique |

In [2]:
import pandas as pd

# Taux de neutres/insatisfaits par modalité : justifie de garder ou d'exclure chaque variable
lignes = []
for col in ["Gender"] + CATEGORICAL_FEATURES:
    t = df.groupby(col)[TARGET].agg(taux_insatisfaits="mean", passagers="size")
    t.index = [f"{col} = {v}" for v in t.index]
    lignes.append(t)
pd.concat(lignes).round(3)

,taux_insatisfaits,passagers
Gender = Female,0.573,52727
Gender = Male,0.561,51177
Customer Type = Loyal Customer,0.523,84923
Customer Type = disloyal Customer,0.763,18981
Type of Travel = Business travel,0.417,71655
Type of Travel = Personal Travel,0.898,32249
Class = Business,0.306,49665
Class = Eco,0.814,46745
Class = Eco Plus,0.754,7494


Ce tableau prouve deux choses :

le genre ne change presque rien (57,3 % contre 56,1 % d’insatisfaits), donc on peut l’exclure sans rien perdre ;
le type de voyage est très parlant (89,8 % contre 41,7 %), d’où l’importance de savoir s’il est connu ou non.

## 2. Méthode de comparaison

**Pourquoi pas test.csv ?** Il représente les passagers du mois prochain : s'en servir pour choisir le modèle rendrait la performance annoncée optimiste et sans valeur. Il n'est utilisé qu'une fois, à la fin (notebook 03).

**Validation croisée à 5 plis sur train.csv.** Train est découpé en 5 parts (même proportion d'insatisfaits dans chacune). Le modèle apprend sur 4 parts et note la 5ᵉ, cinq fois de suite : chaque passager reçoit un score d'un modèle qui ne l'a jamais vu, comme un passager du mois prochain. Tous les modèles et toutes les règles sont notés sur **le même découpage** (`random_state=42`).

**Critère de choix : le gain net, pas l'exactitude.** Les équipes peuvent aborder au plus 10 % des passagers. On classe les passagers par score et on aborde les 10 % en tête. Chaque vrai insatisfait abordé rapporte 1/3 × 1/2 × 600 € − 40 € = **+60 €**, chaque satisfait abordé coûte **−40 €**. L'approche est rentable dès que plus de **40 %** des passagers abordés sont réellement insatisfaits.

**Préparation des données (dans un pipeline).**
- Retard à l'arrivée manquant (310 passagers, 0,3 %) remplacé par le retard au départ : les deux sont quasi identiques (corrélation 0,965). Le calcul se fait ligne par ligne, sans utiliser d'information d'autres passagers.
- Variables catégorielles transformées en colonnes 0/1 ; variables numériques mises à l'échelle pour la régression logistique.
- La préparation est recalculée à chaque pli, uniquement sur les données d'apprentissage, pour éviter toute fuite d'information.

**Reproductibilité.** Toutes les graines aléatoires sont fixées à 42 : deux exécutions donnent exactement les mêmes chiffres.

In [3]:
# Justification du traitement du retard d'arrivée manquant
manquants = df["Arrival Delay in Minutes"].isna().sum()
correlation = df["Departure Delay in Minutes"].corr(df["Arrival Delay in Minutes"])
manquants, round(correlation, 3)

(np.int64(310), np.float64(0.965))

## 3. Règle simple sans modèle (baseline)
Hasard (10 % des passagers) et règle métier.

In [4]:
import numpy as np

rng = np.random.default_rng(RANDOM_STATE)   # hasard reproductible : même tirage à chaque exécution
scores_hasard = rng.random(len(y))          # un score aléatoire par passager

gain_net(y, scores_hasard)

{'passagers_abordes': 10390,
 'insatisfaits_abordes': 5876,
 'precision': 0.5655437921077959,
 'rappel': 0.0997978905891744,
 'gain_net': 172000.0,
 'gain_net_pour_1000': 1655.3741915614412}

In [5]:
# Retard d'arrivée manquant (310 passagers) : on le remplace par le retard au départ,
# les deux sont quasi identiques (corrélation 0,97)
retard = X["Arrival Delay in Minutes"].fillna(X["Departure Delay in Minutes"])

# Règle 1 : aborder les plus gros retards à l'arrivée
regle_retard = gain_net(y, retard)

# Règle 2 : voyage perso d'abord, puis les plus gros retards
perso = (X["Type of Travel"] == "Personal Travel").astype(int)
regle_perso = gain_net(y, perso * 10_000 + retard)

import pandas as pd
baselines = pd.DataFrame(
    [gain_net(y, scores_hasard), regle_retard, regle_perso],
    index=["Hasard", "Règle : plus gros retards", "Règle : voyage perso puis retard"],
)
baselines[["precision", "rappel", "gain_net_pour_1000"]].round(3)

,precision,rappel,gain_net_pour_1000
Hasard,0.566,0.100,1655.374
Règle : plus gros retards,0.642,0.113,2422.428
Règle : voyage perso puis retard,0.956,0.169,5564.752


## 4. Modèle 1 — régression logistique

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Mêmes données que les règles : retard d'arrivée manquant = retard au départ
X_modele = X.copy()
X_modele["Arrival Delay in Minutes"] = retard

# Préparation : nombres mis à la même échelle, catégories transformées en colonnes 0/1
preparation = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])
logistique = make_pipeline(preparation, LogisticRegression(max_iter=1000))

# Validation croisée en 5 parts, toujours le même découpage (random_state)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_logistique = cross_val_predict(
    logistique, X_modele, y, cv=cv, method="predict_proba"
)[:, 1]

resultat_logistique = gain_net(y, scores_logistique)
resultat_logistique

{'passagers_abordes': 10390,
 'insatisfaits_abordes': 9732,
 'precision': 0.936669874879692,
 'rappel': 0.16528813329030723,
 'gain_net': 557600.0,
 'gain_net_pour_1000': 5366.492146596858}

## 5. Modèle 2 — forêt aléatoire / gradient boosting

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Les arbres n'ont pas besoin de mise à l'échelle : les nombres passent tels quels
preparation_arbres = ColumnTransformer([
    ("num", "passthrough", NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])
boosting = make_pipeline(
    preparation_arbres,
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
)

# Même découpage en 5 parts que la logistique : la comparaison est équitable
scores_boosting = cross_val_predict(
    boosting, X_modele, y, cv=cv, method="predict_proba"
)[:, 1]

resultat_boosting = gain_net(y, scores_boosting)
resultat_boosting

{'passagers_abordes': 10390,
 'insatisfaits_abordes': 10049,
 'precision': 0.9671799807507219,
 'rappel': 0.17067205625095536,
 'gain_net': 589300.0,
 'gain_net_pour_1000': 5671.581459809055}

## 6. Comparaison et choix du modèle
Précision dans le top 10 %, gain net pour 1 000 passagers, coût des erreurs, lisibilité.

In [8]:
maximum = gain_net(y, y.values)["gain_net_pour_1000"]   # si on n'abordait que des insatisfaits

comparaison = pd.concat([
    baselines,
    pd.DataFrame(
        [resultat_logistique, resultat_boosting],
        index=["Régression logistique", "Gradient boosting"],
    ),
])
comparaison["% du gain maximal"] = comparaison["gain_net_pour_1000"] / maximum * 100

comparaison[["precision", "rappel", "gain_net_pour_1000", "% du gain maximal"]].round(3)

,precision,rappel,gain_net_pour_1000,% du gain maximal
Hasard,0.566,0.100,1655.374,27.591
Règle : plus gros retards,0.642,0.113,2422.428,40.375
Règle : voyage perso puis retard,0.956,0.169,5564.752,92.749
Régression logistique,0.937,0.165,5366.492,89.445
Gradient boosting,0.967,0.171,5671.581,94.530


## 7. Sans le type de voyage
La compagnie ne connaît peut-être pas le motif du voyage à l'atterrissage : on refait la comparaison sans cette variable.

In [9]:
# Mêmes variables, moins "Type of Travel"
cat_sans = [c for c in CATEGORICAL_FEATURES if c != "Type of Travel"]
X_sans = X_modele[NUMERIC_FEATURES + cat_sans]

logistique_sans = make_pipeline(
    ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_sans),
    ]),
    LogisticRegression(max_iter=1000),
)
boosting_sans = make_pipeline(
    ColumnTransformer([
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_sans),
    ]),
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
)

scores_logistique_sans = cross_val_predict(logistique_sans, X_sans, y, cv=cv, method="predict_proba")[:, 1]
scores_boosting_sans = cross_val_predict(boosting_sans, X_sans, y, cv=cv, method="predict_proba")[:, 1]

# Meilleure règle simple sans le type de voyage : classe Eco d'abord, puis les plus gros retards
eco = (X["Class"] == "Eco").astype(int)

comparaison_sans = pd.DataFrame(
    [
        gain_net(y, scores_hasard),
        regle_retard,
        gain_net(y, eco * 10_000 + retard),
        gain_net(y, scores_logistique_sans),
        gain_net(y, scores_boosting_sans),
    ],
    index=["Hasard", "Règle : plus gros retards", "Règle : Eco puis retard",
           "Régression logistique", "Gradient boosting"],
)
comparaison_sans["% du gain maximal"] = comparaison_sans["gain_net_pour_1000"] / maximum * 100
comparaison_sans[["precision", "rappel", "gain_net_pour_1000", "% du gain maximal"]].round(3)

,precision,rappel,gain_net_pour_1000,% du gain maximal
Hasard,0.566,0.100,1655.374,27.591
Règle : plus gros retards,0.642,0.113,2422.428,40.375
Règle : Eco puis retard,0.874,0.154,4743.802,79.066
Régression logistique,0.855,0.151,4551.317,75.858
Gradient boosting,0.951,0.168,5511.819,91.867


## 8. Ce que fait le modèle retenu (gradient boosting sans type de voyage)

In [10]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X_app, X_val, y_app, y_val = train_test_split(
    X_sans, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
boosting_sans.fit(X_app, y_app)

importance = permutation_importance(
    boosting_sans, X_val, y_val, scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE
)
pd.Series(importance.importances_mean, index=X_sans.columns).sort_values(ascending=False).round(3)

Class                         0.216
Age                           0.064
Customer Type                 0.064
Arrival Delay in Minutes      0.017
Flight Distance               0.009
Departure Delay in Minutes    0.001
dtype: float64

In [11]:
from src.economics import CAPACITY

nb_abordes = int(np.floor(CAPACITY * len(y)))
seuil = np.sort(scores_boosting_sans)[::-1][nb_abordes - 1]
signale = scores_boosting_sans >= seuil      # les 10 % au score le plus élevé

def portrait(masque):
    return [
        X.loc[masque, "Class"].eq("Eco").mean(),
        X.loc[masque, "Customer Type"].eq("disloyal Customer").mean(),
        X.loc[masque, "Age"].median(),
        X.loc[masque, "Age"].lt(18).mean(),
        retard[masque].median(),
        X.loc[masque, "Flight Distance"].median(),
        y[masque].mean(),
    ]

tous = np.ones(len(y), dtype=bool)
pd.DataFrame(
    {"Tous les passagers": portrait(tous), "Passagers signalés": portrait(signale)},
    index=["Part en classe Eco", "Part de clients non fidèles", "Âge médian", "Part de mineurs",
           "Retard médian à l'arrivée (min)", "Distance médiane", "Part réellement insatisfaits"],
).round(3)

,Tous les passagers,Passagers signalés
Part en classe Eco,0.450,0.804
Part de clients non fidèles,0.183,0.612
Âge médian,40.000,33.000
Part de mineurs,0.076,0.237
Retard médian à l'arrivée (min),0.000,13.000
Distance médiane,843.000,602.000
Part réellement insatisfaits,0.567,0.951


## 9. Coût des erreurs et protection des mineurs
le sujet demande que la direction comprenne « ce que coûtent ses erreurs ». Un modèle peut se tromper de deux façons :

Faux positif : on aborde un passager qui était satisfait. On perd 40 € pour rien.
Faux négatif : on n’aborde pas un passager insatisfait. On perd une chance de récupérer 100 € en moyenne.

In [12]:
from src.economics import COST_APPROACH, VALUE_TRUE_POSITIVE

r = gain_net(y, scores_boosting_sans)
vrais_positifs = r["insatisfaits_abordes"]
faux_positifs = r["passagers_abordes"] - vrais_positifs
insatisfaits_non_abordes = int(y.sum()) - vrais_positifs
pour_1000 = 1000 / len(y)

pd.DataFrame({
    "Pour 1 000 passagers arrivants": [
        r["passagers_abordes"] * pour_1000,
        vrais_positifs * pour_1000,
        faux_positifs * pour_1000,
        faux_positifs * COST_APPROACH * pour_1000,
        insatisfaits_non_abordes * pour_1000,
    ]
}, index=[
    "Passagers abordés",
    "dont vraiment insatisfaits",
    "dont satisfaits (faux positifs)",
    "Coût des faux positifs (€)",
    "Insatisfaits non abordés",
]).round(1)

,Pour 1 000 passagers arrivants
Passagers abordés,100.0
dont vraiment insatisfaits,95.1
dont satisfaits (faux positifs),4.9
Coût des faux positifs (€),195.2
Insatisfaits non abordés,471.6


les faux positifs sont presque négligeables. Les insatisfaits non abordés ne sont pas une erreur du modèle, c’est la capacité des équipes : 57 % des passagers sont insatisfaits, mais on ne peut en aborder que 10 %. Même un modèle parfait en laisserait plus de 80 % sans contact. Le levier principal pour eux, ce sont les améliorations de service recommandées par Léa, pas le modèle.

Faut-il utiliser toute la capacité de 10 % ?

In [13]:
pd.DataFrame(
    [gain_net(y, scores_boosting_sans, capacity=c) for c in [0.02, 0.05, 0.08, 0.10]],
    index=["2 %", "5 %", "8 %", "10 %"],
)[["precision", "gain_net_pour_1000"]].round(3)

,precision,gain_net_pour_1000
2 %,0.983,1165.306
5 %,0.970,2850.708
8 %,0.959,4471.628
10 %,0.951,5511.819


Les trois options pour les mineurs : 

23,7 % des passagers signalés sont mineurs. Signaler un enfant au personnel pose un problème de protection des données. On chiffre chaque option pour que Côme et la direction choisissent en connaissant le coût.

In [14]:
mineur = (X["Age"] < 18).values

# Option 1 : même modèle, mais on ne signale jamais un mineur (score mis tout en bas)
option_exclusion = gain_net(y, np.where(mineur, -np.inf, scores_boosting_sans))

# Option 2 : on retire l'âge du modèle
num_sans_age = [c for c in NUMERIC_FEATURES if c != "Age"]
boosting_sans_age = make_pipeline(
    ColumnTransformer([
        ("num", "passthrough", num_sans_age),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_sans),
    ]),
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
)
scores_sans_age = cross_val_predict(
    boosting_sans_age, X_sans[num_sans_age + cat_sans], y, cv=cv, method="predict_proba"
)[:, 1]
option_sans_age = gain_net(y, scores_sans_age)

def part_mineurs(scores):
    nb = int(np.floor(CAPACITY * len(y)))
    return mineur[np.argsort(-scores, kind="stable")[:nb]].mean()

options = pd.DataFrame(
    [gain_net(y, scores_boosting_sans), option_exclusion, option_sans_age],
    index=["Modèle tel quel", "Option 1 : mineurs exclus du signalement", "Option 2 : âge retiré du modèle"],
)
options["part de mineurs signalés"] = [
    part_mineurs(scores_boosting_sans),
    0.0,
    part_mineurs(scores_sans_age),
]
options[["precision", "gain_net_pour_1000", "part de mineurs signalés"]].round(3)

,precision,gain_net_pour_1000,part de mineurs signalés
Modèle tel quel,0.951,5511.819,0.237
Option 1 : mineurs exclus du signalement,0.939,5385.741,0.000
Option 2 : âge retiré du modèle,0.897,4972.860,0.116


La conclusion, contre-intuitive mais solide :

Retirer l’âge ne règle pas le problème. Le modèle signale encore 11,6 % de mineurs, parce qu’il les repère autrement (classe, fidélité, distance), et il perd 539 €.
Garder l’âge et exclure les mineurs du signalement coûte seulement 126 € pour 1 000 passagers, soit 2 % du gain, et garantit qu’aucun enfant n’est signalé.
Pour Côme : l’âge sert ici à protéger les mineurs, pas à cibler qui que ce soit. On pourra aussi proposer d’aborder l’adulte qui accompagne l’enfant (option 3), mais nos données ne permettent pas de la chiffrer, car on ne sait pas qui voyage avec qui.

→ Recommandation : gradient boosting sans le type de voyage, avec les mineurs exclus du signalement. Environ 5 386 € pour 1 000 passagers arrivants, 94 % de précision.